# libraries and data

In [76]:
import dill as pickle
from collections import Counter, defaultdict
from unidecode import unidecode
from scipy.sparse import csr_matrix
from scipy.stats import fisher_exact
import numpy as np

In [ ]:
fn = '../../rsc/DRC/corpus_doreco.p' # change this to wherever the pickle file lives
corpus = pickle.load(open(fn, 'rb')) # reads in the full corpus

# demonstrating the lay out of the corpus

In [2]:
# the corpus consists of a dictionary with the language names as keys
print(corpus.keys())

dict_keys(['anal1239.csv', 'apah1238.csv', 'arap1274.csv', 'bain1259.csv', 'beja1238.csv', 'bora1263.csv', 'cabe1245.csv', 'cash1254.csv', 'dolg1241.csv', 'even1259.csv', 'goem1240.csv', 'goro1270.csv', 'hoch1243.csv', 'jeha1242.csv', 'jeju1234.csv', 'kaka1265.csv', 'kama1351.csv', 'kark1256.csv', 'komn1238.csv', 'ligh1234.csv', 'lowe1385.csv', 'movi1243.csv', 'ngal1292.csv', 'nisv1234.csv', 'nngg1234.csv', 'nort2641.csv', 'nort2875.csv', 'orko1234.csv', 'pnar1238.csv', 'port1286.csv', 'resi1247.csv', 'ruul1235.csv', 'sadu1234.csv', 'sanz1248.csv', 'savo1255.csv', 'sout2856.csv', 'sout3282.csv', 'stan1290.csv', 'sumi1235.csv', 'svan1243.csv', 'taba1259.csv', 'teop1238.csv', 'texi1237.csv', 'trin1278.csv', 'tsim1256.csv', 'urum1249.csv', 'vera1241.csv', 'warl1254.csv', 'yong1270.csv', 'yuca1254.csv', 'yura1255.csv'])


In [5]:
# for every language key, the value is again a dictionary, with the elicitation session files as the keys (first 5 printed here for kama1351)
# with every elicitation file again having the 'lines' as their keys
lg = 'kama1351.csv'
print(list(corpus[lg].keys())[:5])
file = 'doreco_kama1351_PKZ_1964_SU0204'
print(list(corpus[lg][file].keys())[:5])

['doreco_kama1351_PKZ_1964_SU0204', 'doreco_kama1351_PKZ_1964_SU0205', 'doreco_kama1351_PKZ_1964_SU0206', 'doreco_kama1351_PKZ_1964_SU0207', 'doreco_kama1351_PKZ_1964_SU0208']
['0001_DoReCo_doreco_kama1351_PKZ_1964_SU0204', '0002_DoReCo_doreco_kama1351_PKZ_1964_SU0204', '0003_DoReCo_doreco_kama1351_PKZ_1964_SU0204', '0004_DoReCo_doreco_kama1351_PKZ_1964_SU0204', '0005_DoReCo_doreco_kama1351_PKZ_1964_SU0204']


In [9]:
# every line key has a value that's (again) a dictionary, containing several fields: gloss, free translation (ft), text (tx), and spacied free translation (spc)
line = '0005_DoReCo_doreco_kama1351_PKZ_1964_SU0204'
for k,v in corpus[lg][file][line].items():
    print(k,v)

gloss [{'w': 'iʔ', 'wid': 'w5', 'M': ['e', 'ʔ'], 'G': ['NEG.AUX', 'IMP.2SG']}, {'w': 'kudonzaʔ', 'wid': 'w6', 'M': ['kudo', 'nzə', 'ʔ'], 'G': ['scold', 'DES', 'CNG']}]
ft Don't curse.
tx iʔ kudonzaʔ
spc [{'text': 'Do', 'i': 0, 'idx': 0, 'lemma': 'do', 'pos': 'AUX', 'tag': 'VB', 'head.i': 2, 'dep': 'aux'}, {'text': "n't", 'i': 1, 'idx': 2, 'lemma': 'not', 'pos': 'PART', 'tag': 'RB', 'head.i': 2, 'dep': 'neg'}, {'text': 'curse', 'i': 2, 'idx': 6, 'lemma': 'curse', 'pos': 'VERB', 'tag': 'VB', 'head.i': 2, 'dep': 'ROOT'}, {'text': '.', 'i': 3, 'idx': 11, 'lemma': '.', 'pos': 'PUNCT', 'tag': '.', 'head.i': 2, 'dep': 'punct'}]


In [13]:
# the middle two are self-explanatory, but the first and fourth warrant further explanation
# the gloss is a list of dictionaries, with every dictionary containing
#   * the [w]ord
#   * the [w]ord [id] from the spreadsheets (for identification)
#   * the [M]orphological structure, as a list of morphemes
#   * the [G]loss of the morphological structure
# the demo below prints the two words in our present sentence
for g in corpus[lg][file][line]['gloss']:
    print(g)

{'w': 'iʔ', 'wid': 'w5', 'M': ['e', 'ʔ'], 'G': ['NEG.AUX', 'IMP.2SG']}
{'w': 'kudonzaʔ', 'wid': 'w6', 'M': ['kudo', 'nzə', 'ʔ'], 'G': ['scold', 'DES', 'CNG']}


In [15]:
# the fourth key, 'spc' contains the parse of the free translation 
# (sometimes: a google translate of the free translation if it's in Spanish, Malay, Russian etc)
# the value for 'spc' is a list of the English tokens, with the following information:
#  * 'text': the character string
#  * 'i': the token index
#  * 'idx': the character index of the first token of the character
#  * 'lemma': the lemma
#  * 'pos': the grammatical category (coarse grained)
#  * 'tag': the grammatical category (fine grained; Penn treebank tagset)
#  * 'head.i': the 'i' value of the head of the token in the dependency parse
#  * 'dep': the dependency relation of the current token to its head
for s in corpus[lg][file][line]['spc']:
    print(s)

{'text': 'Do', 'i': 0, 'idx': 0, 'lemma': 'do', 'pos': 'AUX', 'tag': 'VB', 'head.i': 2, 'dep': 'aux'}
{'text': "n't", 'i': 1, 'idx': 2, 'lemma': 'not', 'pos': 'PART', 'tag': 'RB', 'head.i': 2, 'dep': 'neg'}
{'text': 'curse', 'i': 2, 'idx': 6, 'lemma': 'curse', 'pos': 'VERB', 'tag': 'VB', 'head.i': 2, 'dep': 'ROOT'}
{'text': '.', 'i': 3, 'idx': 11, 'lemma': '.', 'pos': 'PUNCT', 'tag': '.', 'head.i': 2, 'dep': 'punct'}


# querying with these data

In [18]:
# querying glosses as a whole
# we can use all this information to retrieve all instances of morphemes containing a standardize marking like 'PERF' or 'PRF'
# and counts which morphemes we find associated with them
# (which is likely not exhaustive; only a subset of languages has explicit markers: bora, jeha, kaka, nort)
# you can tell they need some post-processing too -- bora '-jucoo' and 'jucoo' should be merged
for lg in corpus:
    lg_count = Counter()
    for file in corpus[lg]:
        for line in corpus[lg][file]:
            line_obj = corpus[lg][file][line]
            for g in line_obj['gloss']:
                for morph,gloss in zip(g['M'], g['G']): # this iterates over all pairs of morphemes and their glosses
                    if 'PERF' in gloss or 'PRF' in gloss: # tests if the gloss CONTAINS the relevant strings
                        lg_count[morph, gloss] += 1
    print(lg, lg_count.most_common(10))
                        

anal1239.csv []
apah1238.csv []
arap1274.csv [(('nii-', 'IMPERF'), 165), (('woow', 'now.PERF'), 115), (("nih'ii-", 'PAST.IMPERF'), 87), (('ii-', 'IMPERF'), 69), (("nii'-", 'when.IMPERF'), 36), (('niis-', 'how.IMPERF'), 30), (('tohuu-', 'since.IMPERF'), 29), (("'ii-", 'IMPERF'), 17), (('hiihoow-', '3.IMPERF.NEG'), 15), (("he'ih'ii-", 'NARRPAST.IMPERF'), 14)]
bain1259.csv []
beja1238.csv []
bora1263.csv [(('-jucoo', 'PERF'), 73), (('jucoo', 'PERF'), 16), (('-jucoo.-?', 'PERF'), 1)]
cabe1245.csv []
cash1254.csv [(('a', 'PERF'), 1)]
dolg1241.csv []
even1259.csv []
goem1240.csv []
goro1270.csv []
hoch1243.csv []
jeha1242.csv [(('sudah', 'PERF'), 3), (('sudah...', 'PERF'), 1)]
jeju1234.csv [(('na', 'AUX.PERF'), 37)]
kaka1265.csv [(('báti', 'PRF'), 498), (('máa', 'NEG.PRF'), 112), (('ti', 'PRF'), 9), (('n', '1SGGPRF'), 1), (('ta', 'PRF'), 1)]
kama1351.csv []
kark1256.csv []
komn1238.csv []
ligh1234.csv []
lowe1385.csv []
movi1243.csv []
ngal1292.csv []
nisv1234.csv []
nngg1234.csv []
nort2641

In [47]:
# we can also look for instances where English uses a perfective and study examples (three per language)
for lg in corpus:
    top_n = 3
    for file in corpus[lg]:
        for line in corpus[lg][file]:
            if top_n <= 0: continue
            line_obj = corpus[lg][file][line]
            for s in line_obj['spc']:
                if s['tag'] == 'VBN' and next((True for si in line_obj['spc'] if si['lemma'] == 'have' and si['head.i'] == s['i']),False):
                    # this condition checks if the word is a participle (VBN) and if it has a dependent whose lemma is have -- very much first pass perfect detection
                    print(lg, line_obj['ft'], '\n', line_obj['tx'], '\n', ' '.join('%s/%s' % (m, g) for G in line_obj['gloss'] for m,g in zip(G['M'], G['G'])), '\n')
                    top_n -= 1

anal1239.csv Nobody has seen the dung of the fish from the deep sea 
 du̠ralha ngha vana e to aku̠ra thimi 
  

anal1239.csv we should have put it earlier 
 rowlsa napha doh hrang varende 
  

anal1239.csv The buyer had also sold the cow to someone else 
 mi rinnu jol thunu 
  

apah1238.csv he has passed away, so now 
 meninggal atisimure ketiare 
 meninggal/pass.away at/become.3s -tisi/PST =mu/SIT =te/FOC ketia/now =te/FOC 

apah1238.csv he has slept 
 tu nohoruk lahareg 
 tu/DET noho/sleep -tuk/PROG laha/3s.IM.PST -teg/do.SS.PRIOR 

apah1238.csv he has woken up 
 in atuk ari 
 in/stand at/become -tuk/PROG ari/DEM.s 

arap1274.csv I have heard that you elk are very fast. 
 neniitowoo3e3enee toh'eso'ooneehek 
 ne-/1S niitowoo3/hear.about -e3enee/1S.2PL toh-/since 'eso'oo/fast -nee/2PL -hek/SUBJ 

arap1274.csv And then after, I guess, he had slaughtered [the elk], then he went down [to the base of the cliff]. 
 'oh he'ne'iisp wootii noo-noo'oohoot noh he'ne'hoowuseet 
 'oh/but.and he'n

# automatically extracting (first pass) with these data

if we know where English has perfects, we can determine which morphemes and glosses have the strongest association with the presence of a perfect in English

In [75]:
## functions for fragment extraction based on the Liu et al 2023 algorithm (see my papers for cites)

def get_fragments(w, min_len=2, max_len=7, yield_self=True):
    """
    takes a string *w* and extracts all substrings of minimal length *min_len* and maximal length *max_len*
    """
    if yield_self and len(w) > max_len: yield w
    for i in range(len(w)):
        for j in range(i+min_len, min(i+max_len+1,len(w)+1)):
            if (not (i == 0 or j == len(w))) or (j-i) >= min_len+1:
                yield w[i:j]

def get_all_fragments(F, split_words = True, frequency_threshold = 1):
    """
    given a document stored in a dictionary *tbd*, mapping an identifier key 
    onto a string containing the text, and a set of *all_verses*
    (the shared identifier keys between tbd and the source document(,
    this function returns a sparse matrix *fragments* of identifier key (rows) 
    to substrings of the text (column), with the matrix being True if the fragment
    occurs for that identifier key and False otherwise.
    as well as dictionaries for the identification
    of the rows and columns.
    (memory/computation efficient format, but a bit densely written)
    """
    wordcount = Counter((w for l in F for w in l))
    if split_words: 
        word_fragments = {w : set(get_fragments('^%s$' % unidecode(w).lower())) for w in wordcount.keys() }
    else: 
        word_fragments = {w : {unidecode(w).lower()} for w in wordcount.keys() }
    
    fragment_count = Counter((f for w,F in word_fragments.items() for f in F if f != '' for i in range(wordcount[w])))
    fragment_ixx = {f:i for i,(f,c) in enumerate(fragment_count.most_common()) if c >= frequency_threshold }
    #
    R,C = [], []
    for line_ix, line_f in enumerate(F):
        if len(line_f) == 0: continue
        line_frags = list(map(lambda f : fragment_ixx[f],
                              filter(lambda f : f in fragment_ixx,
                                     set.union(*map(lambda w : word_fragments[w], line_f)))))
        R.extend([line_ix]*len(line_frags))
        C.extend(line_frags)
    fragments = csr_matrix((np.ones(len(R)), (R,C)), dtype=bool, shape = (len(F),max(C)+1))
    return fragments, fragment_ixx

In [74]:
def extract_tes(pos_ixx, fragments, fixx, coverage = 0.95, min_trans = 0.05, min_backtrans = 0.25, verbose=False):
    """
    implements one forward pass of the Liu et al. (2023) approach.
    Takes a list *pos* of positive instances (row identifiers for fragments)
    As well as the sparse matrix *fragments* and the two dictionaries
    (vixx -- verse_ixx and fixx -- fragment_ixx).
    The parameters determine the fragments considered in the extraction: the iteration keeps going until
    either no good fragments can be found (significance under .001) or the *coverage* has been reached.
    *min_trans* is a float [0,1] that filters out all fragments that occur in hit verses in less than min_trans
    proportion of all hit verses.
    """
    neg_ixx = list(set(range(fragments.shape[0]))-set(pos_ixx))
    flist = [None] * len(fixx)
    for k,v in fixx.items():
        flist[v] = k
    #sorted(fixx, key = lambda k : fixx[k])
    #
    pos_tot_orig = pos_tot = len(pos_ixx)
    neg_tot = len(neg_ixx)
    #
    pos_ct = fragments[pos_ixx].sum(0).A[0]
    neg_ct = fragments[neg_ixx].sum(0).A[0]
    #
    good_fragments = np.where((pos_ct >= 1) & ((pos_ct/pos_tot_orig) >= min_trans) &
                             (pos_ct/(pos_ct+neg_ct) >= min_backtrans))[0]
    string_props = [(f[0] == '^', f[-1] == '$', len(f)) for f in flist]
                    #len(re.sub('[.*]', '', f)))
    #
    hits = defaultdict(lambda : [])
    ct = 0
    fe_scores = {}
    while len(pos_ixx) >= (1-coverage) * pos_tot_orig:
        ct += 1
        #
        # GET BEST
        assoc_scores = Counter()
        for f in good_fragments:
            table = ((pos_ct[f],pos_tot-pos_ct[f]),(neg_ct[f],neg_tot-neg_ct[f]))
            try: fe_score = fe_scores[table]
            except KeyError: fe_score = fe_scores[table] = -np.log(fisher_exact(table, alternative='greater')[1])
            assoc_scores[f] = (fe_score, string_props[f])
        best, best_score = next((x for x in assoc_scores.most_common(1)),(None,None))
        if best == None or best_score[0] < -np.log(5e-2):
            break
        # print([flist[k] for k,v in assoc_scores.most_common(10)])
        #
        # UPDATE
        new_pos_ixx = []
        for pos_v in pos_ixx:
            if fragments[pos_v,best] > 0: hits[flist[best]].append(pos_v)
            else: new_pos_ixx.append(pos_v)
        #
        pos_ixx = new_pos_ixx
        pos_tot = len(pos_ixx)
        pos_ct = fragments[pos_ixx].sum(0).A[0]
        neg_ct = fragments[neg_ixx].sum(0).A[0]
        #
        good_fragments = np.where((pos_ct >= 1) & (pos_ct/pos_tot_orig >= min_trans) &
                                    (pos_ct/(pos_ct+neg_ct) >= min_backtrans))[0]
        if verbose: print(ct, flist[best])
    return hits

In [66]:
# first, create a list of lists with target language morphemes
# as well as a list of lists of glosses
# note that this tells us which languages don't have glossed lines; they're removed from consideration
morphemes = {}
glosses = {}
for lg in corpus:
    morphemes[lg] = []
    glosses[lg] = []
    for file in corpus[lg]:
        for line in corpus[lg][file]:
            line_obj = corpus[lg][file][line]
            glosses_l = [g for W in line_obj['gloss'] for g in W['G']]
            morphemes_l = [m for W in line_obj['gloss'] for m in W['M']]
            if len(glosses_l) == 0 or len(morphemes_l) == 0: continue
            glosses[lg].append(glosses_l)
            morphemes[lg].append(morphemes_l)
    print(lg, len(glosses[lg]))
    if len(glosses[lg]) == 0:
        del glosses[lg]

anal1239.csv 0
apah1238.csv 2422
arap1274.csv 3134
bain1259.csv 2341
beja1238.csv 6306
bora1263.csv 3798
cabe1245.csv 2052
cash1254.csv 1215
dolg1241.csv 2425
even1259.csv 2311
goem1240.csv 2528
goro1270.csv 3814
hoch1243.csv 971
jeha1242.csv 769
jeju1234.csv 1937
kaka1265.csv 4366
kama1351.csv 8949
kark1256.csv 0
komn1238.csv 7458
ligh1234.csv 0
lowe1385.csv 0
movi1243.csv 1655
ngal1292.csv 771
nisv1234.csv 1675
nngg1234.csv 6198
nort2641.csv 834
nort2875.csv 2661
orko1234.csv 2224
pnar1238.csv 930
port1286.csv 1124
resi1247.csv 0
ruul1235.csv 2902
sadu1234.csv 0
sanz1248.csv 404
savo1255.csv 774
sout2856.csv 1545
sout3282.csv 0
stan1290.csv 0
sumi1235.csv 2468
svan1243.csv 0
taba1259.csv 632
teop1238.csv 1962
texi1237.csv 2817
trin1278.csv 1841
tsim1256.csv 0
urum1249.csv 2295
vera1241.csv 1583
warl1254.csv 0
yong1270.csv 0
yuca1254.csv 0
yura1255.csv 0


In [48]:
# this retrieves all row indices of the list of lists retrieved in the previous step of
# lines where a perfect was present in the translation
perfects = {}
for lg in corpus:
    if lg not in glosses: continue
    perfects[lg] = []
    i = 0
    for file in corpus[lg]:
        for line in corpus[lg][file]:
            line_obj = corpus[lg][file][line]
            glosses_l = [g for W in line_obj['gloss'] for g in W['G']]
            morphemes_l = [m for W in line_obj['gloss'] for m in W['M']]
            if len(glosses_l) == 0 or len(morphemes_l) == 0: continue
            for s in line_obj['spc']:
                if s['tag'] == 'VBN' and next((True for si in line_obj['spc'] if si['lemma'] == 'have' and si['head.i'] == s['i']),False):
                    # if len(perfects[lg]) < 5: print(['/'.join(map(str, [w['text'], w['i'], w['head.i'], w['dep']])) for w in line_obj['spc']])
                    # this condition checks if the word is a participle (VBN) and if it has a dependent whose lemma is have -- very much first pass perfect detection
                    perfects[lg].append(i)
            i += 1

['he/0/2/nsubj', 'has/1/2/aux', 'passed/2/2/ROOT', 'away/3/2/prt', ',/4/2/punct', 'so/5/2/cc', 'now/6/2/advmod']
['he/0/2/nsubj', 'has/1/2/aux', 'slept/2/2/ROOT']
['he/0/2/nsubj', 'has/1/2/aux', 'woken/2/2/ROOT', 'up/3/2/prt']
['Do/0/2/aux', 'I/1/2/nsubj', 'have/2/2/ROOT', 'to/3/4/aux', 'repeat/4/2/xcomp', 'what/5/9/dobj', 'I/6/9/nsubj', 'have/7/9/aux', 'ever/8/9/advmod', 'told/9/4/ccomp', '?/10/2/punct']
['I/0/1/nsubj', 'was/1/1/ROOT', 'the/2/3/det', 'story/3/1/attr', 'that/4/6/nsubj', 'had/5/6/aux', 'happened/6/3/relcl', 'in/7/6/prep', 'the/8/9/det', 'winek/9/7/pobj']
['I/0/2/nsubj', 'have/1/2/aux', 'heard/2/2/ROOT', 'that/3/6/mark', 'you/4/6/nsubj', 'elk/5/4/appos', 'are/6/2/ccomp', 'very/7/8/advmod', 'fast/8/6/acomp', './9/2/punct']
['And/0/9/cc', 'then/1/9/advmod', 'after/2/9/mark', ',/3/5/punct', 'I/4/5/nsubj', 'guess/5/9/parataxis', ',/6/9/punct', 'he/7/9/nsubj', 'had/8/9/aux', 'slaughtered/9/17/ccomp', '[/10/9/punct', 'the/11/12/det', 'elk/12/9/dobj', ']/13/9/punct', ',/14/17/p

In [50]:
# prints number of English perfects per language
for k,v in perfects.items():
    print(k, len(v))

apah1238.csv 30
arap1274.csv 154
bain1259.csv 187
beja1238.csv 315
bora1263.csv 446
cabe1245.csv 57
cash1254.csv 79
dolg1241.csv 193
even1259.csv 35
goem1240.csv 248
goro1270.csv 212
hoch1243.csv 42
jeha1242.csv 16
jeju1234.csv 53
kaka1265.csv 396
kama1351.csv 97
komn1238.csv 145
movi1243.csv 78
ngal1292.csv 8
nisv1234.csv 20
nngg1234.csv 141
nort2641.csv 95
nort2875.csv 48
orko1234.csv 69
pnar1238.csv 24
port1286.csv 27
ruul1235.csv 230
sanz1248.csv 17
savo1255.csv 29
sout2856.csv 63
sumi1235.csv 122
taba1259.csv 42
teop1238.csv 107
texi1237.csv 46
trin1278.csv 92
urum1249.csv 139
vera1241.csv 120


In [82]:
# now we can extract the most strongly associated morphemes and glosses per language
# these can in turn be used to query in ways described above and see if they make sense
# (for one thing, many glosses are lowercase, and it looks like there are strong anisomorphies
# in when these markers are applied compared to English -- cf Bora)
# there's also some parameter tweakign to be done here and some defining the query on the English side
# it remains just a heuristic to get going on individual languages.

for lg in corpus:
    if lg not in glosses: continue
    print(lg)
    for targname, target in [['glosses', glosses[lg]], ['morphemes', morphemes[lg]]]:
        F, fixx = get_all_fragments(target, split_words=False)
        translations = extract_tes(perfects[lg], F, fixx, coverage = 0.95, min_trans = 0.05, min_backtrans = 0.05, verbose=False)
        translation_count = Counter({k:len(v) for k,v in translations.items()})
        print(targname, translation_count.most_common(10))
    print()

apah1238.csv
glosses [('say.2s.im.pst', 3), ('bride.price', 2), ('become.3s', 2), ('1s.im.pst', 2), ('do.1s.rem.pst', 2), ('come.3s.im.pst', 2), ('1p.pst', 2)]
morphemes [('waha', 5), ('ehen', 3), ('-ihi', 3), ('henggo', 2), ('ete', 2), ('-tisi', 2), ('u', 2), ('hinahan', 2)]

arap1274.csv
glosses [('3.s', 27), ('to.here', 21), ('now.perf', 20), ('3s', 18), ('2s', 10)]
morphemes [('woow', 20), ('cih-', 20), ('-oot', 11), ("-o'", 9), ("-no'", 9), ('toh-', 8)]

bain1259.csv
glosses [('perf', 100), ('rel', 33), ('pass', 13)]
morphemes [('-i', 97), ('-ne', 17), ('leeb', 11), ('n', 11), ('-ang', 10)]

beja1238.csv
glosses [('cvb.mnr', 81), ('cvb.seq', 76), ('mnr.indf.m.acc', 38)]


/tmp/ipykernel_802865/637961750.py:59: RuntimeWarning: invalid value encountered in divide
  (pos_ct/(pos_ct+neg_ct) >= min_backtrans))[0]


morphemes [('=b', 101), ('-ti:t', 42), ('-e:ti:t', 31), ('-n', 18)]

bora1263.csv
glosses [('inan', 229), ('rec', 113), ('1pl.excl', 39)]
morphemes [('-ne', 286), ('muha', 50), ('ne', 43)]

cabe1245.csv
glosses [('sub', 11), ('ag', 8), ('cmpl', 6), ('which', 5), ('devour', 4), ('now', 3), ('from', 3)]
morphemes [('menetse', 6), ('boi', 5), ('kata', 4), ('ke', 4), ('bla', 3), ('pa', 3), ('jana', 3)]

cash1254.csv
glosses [('noct', 10), ('top', 10), ('already', 8), ('without', 6), ('still', 4), ('fire', 4), ('att', 4), ('x', 4)]
morphemes [('xina', 11), ('dan', 10), ('di', 8), ('tan', 8), ('txi', 5), ('xu', 5), ('ka', 5), ('muka', 4), ('uma', 4)]

dolg1241.csv
glosses [('ptcp.pst', 46), ('pst', 31), ('pst2.3sg', 31), ('1sg.nom', 22), ('pst2', 17), ('after', 13)]
morphemes [('bit', 80), ('but', 49), ('e', 29)]

even1259.csv
glosses [('already', 14), ('become', 6), ('cond', 2), ('for a long time', 2)]
morphemes [('uz@', 12), ('teper', 2), ('-mc@', 2), ('ang@', 2)]

goem1240.csv
glosses [('

# end